In [3]:
%pip install pandas scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
from pathlib import Path

# Path setup based on your exact VS Code folder layout
ROOT_DIR = Path.cwd().parent
DATA_DIR = ROOT_DIR / "data" / "NSL-KDD"
MODELS_DIR = ROOT_DIR / "models"

TRAIN_FILE = DATA_DIR / "KDDTrain+.txt"
TEST_FILE = DATA_DIR / "KDDTest+.txt"

print("================== LOCAL PATH VERIFICATION ==================")
print("✅ Training file found on your laptop:", TRAIN_FILE.exists())
print("✅ Testing file found on your laptop :", TEST_FILE.exists())
print("=============================================================")

if not TRAIN_FILE.exists() or not TEST_FILE.exists():
    raise FileNotFoundError(f"❌ Could not find files! Check your folder layout: {DATA_DIR}")

================== LOCAL PATH VERIFICATION ==================
✅ Training file found on your laptop: True
✅ Testing file found on your laptop : True


In [15]:
import pandas as pd

COLUMNS = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
    "wrong_fragment","urgent","hot","num_failed_logins","logged_in",
    "num_compromised","root_shell","su_attempted","num_root",
    "num_file_creations","num_shells","num_access_files",
    "num_outbound_cmds","is_host_login","is_guest_login",
    "count","srv_count","serror_rate","srv_serror_rate",
    "rerror_rate","srv_rerror_rate","same_srv_rate",
    "diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate",
    "dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate",
    "dst_host_srv_rerror_rate","label","difficulty"
]

SELECTED_FEATURES = [
    "duration",
    "src_bytes",
    "dst_bytes",
    "count"
]

print("Loading NSL-KDD dataset...")

train_df = pd.read_csv(TRAIN_FILE, names=COLUMNS)
test_df = pd.read_csv(TEST_FILE, names=COLUMNS)

X_train = train_df[SELECTED_FEATURES].copy()
X_test = test_df[SELECTED_FEATURES].copy()

y_train = (train_df["label"] != "normal").astype(int)
y_test = (test_df["label"] != "normal").astype(int)

print("Train Shape:", X_train.shape)
print("Test Shape :", X_test.shape)

X_train.head()

Loading NSL-KDD dataset...
Train Shape: (125973, 4)
Test Shape : (22544, 4)


,duration,src_bytes,dst_bytes,count
0,0,491,0,2
1,0,146,0,13
2,0,0,0,123
3,0,232,8153,5
4,0,199,420,30


In [16]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

print("Training Random Forest...")

model.fit(X_train_scaled, y_train)

print("Training Complete.")

Training Random Forest...
Training Complete.


In [17]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

y_pred = model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

cm = confusion_matrix(y_test, y_pred)

print("\n================ RESULTS ================")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


================ RESULTS ================
Accuracy : 0.7441
Precision: 0.9243
Recall   : 0.5996
F1 Score : 0.7274

Confusion Matrix:
[[9081  630]
 [5138 7695]]

Classification Report:
              precision    recall  f1-score   support

           0       0.64      0.94      0.76      9711
           1       0.92      0.60      0.73     12833

    accuracy                           0.74     22544
   macro avg       0.78      0.77      0.74     22544
weighted avg       0.80      0.74      0.74     22544



In [18]:
import json
import joblib

MODELS_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODELS_DIR / "model.joblib"
scaler_path = MODELS_DIR / "scaler.joblib"
features_path = MODELS_DIR / "nslkdd_4f_features.json"
metrics_path = MODELS_DIR / "model_metrics.json"

joblib.dump(model, model_path)
joblib.dump(scaler, scaler_path)

with open(features_path, "w") as f:
    json.dump(SELECTED_FEATURES, f)

metrics = {
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1_score": float(f1),
    "confusion_matrix": cm.tolist(),
    "train_samples": int(len(X_train)),
    "test_samples": int(len(X_test)),
    "model_type": "Random Forest",
    "estimators": 200,
    "selected_features": SELECTED_FEATURES,
    "feature_importances": [
        float(x)
        for x in model.feature_importances_
    ]
}

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=4)

print("Saved:")
print(model_path)
print(scaler_path)
print(features_path)
print(metrics_path)

Saved:
c:\Users\Mathuravan\Documents\IIT\Final year\FYP\FYP Project\CyberXAI\models\model.joblib
c:\Users\Mathuravan\Documents\IIT\Final year\FYP\FYP Project\CyberXAI\models\scaler.joblib
c:\Users\Mathuravan\Documents\IIT\Final year\FYP\FYP Project\CyberXAI\models\nslkdd_4f_features.json
c:\Users\Mathuravan\Documents\IIT\Final year\FYP\FYP Project\CyberXAI\models\model_metrics.json


In [19]:
sample = {
    "duration": 12,
    "src_bytes": 7000,
    "dst_bytes": 4000,
    "count": 30
}

input_df = pd.DataFrame([sample])

input_scaled = scaler.transform(input_df)

prediction = model.predict(input_scaled)[0]
probability = model.predict_proba(input_scaled)[0]

label = (
    "Attack / Anomaly Detected"
    if prediction == 1
    else "Normal Safe Traffic"
)

print("\n================ TEST ================")
print("Prediction :", label)
print("Confidence :", probability[prediction])
print("======================================")


================ TEST ================
Prediction : Normal Safe Traffic
Confidence : 0.755
